# Clean bondwire dataset from separated labels

Build a COCO dataset from the clean per-category bondwire labels in `Bent-Wire-ALU-Train/20250512_parted_thin_flat`.

File naming convention:
- `*_original.png`        → image
- `*_lblthin.png`         → thin wire mask
- `*_lblthick.png`        → thick wire mask
- `*_gatepads.png`        → gate pad mask
- `*_combined_labels.png` → composite mask (for sanity checks)
- `*_details.json`        → metadata (optional)

Output layout (per category):
```
<output_root>/
  thick_wire/
    {train,test}/{images,masks}/
    {train,test}/annotation.json
  thin_wire/
    {train,test}/{images,masks}/
    {train,test}/annotation.json
  gate_pad/
    {train,test}/{images,masks}/
    {train,test}/annotation.json
```

Then merge the three categories into a single multi-category COCO dataset and run round-trip verification.

In [ ]:
#| default_exp preprocessing.clean_bondwire_dataset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import json
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from fastcore.all import *

In [ ]:
#| exporti
from sam3_fine_tune.preprocessing.coco_format import form_coco_dataset, save_ann_file


In [ ]:
#| export

# Local copy of merge_datasets_multi so the notebook doesn't depend on nbdev-exported modules.
def _merge_datasets_multi(datasets: list, output_path: str):
    """Merge multiple COCO datasets into one multi-class dataset."""
    for split in ["train", "test"]:
        out_split = Path(output_path) / split
        out_split.mkdir(parents=True, exist_ok=True)
        (out_split / "images").mkdir(exist_ok=True)

        categories = [
            {"id": i + 1, "name": ds["category_name"]} for i, ds in enumerate(datasets)
        ]
        merged = {
            "info": {"description": "Clean bondwire dataset", "version": "1.0", "year": 2025},
            "images": [],
            "categories": categories,
            "annotations": [],
        }

        img_id_counter = 1
        ann_id_counter = 1

        for ds_idx, ds_config in enumerate(datasets):
            ds_path = Path(ds_config["path"])
            prefix = ds_config["prefix"]
            category_id = ds_idx + 1

            ann_path = ds_path / split / "annotation.json"
            if not ann_path.exists():
                print(f"  Skipping {split} - {ds_config['category_name']}: no annotation.json")
                continue

            with open(ann_path) as f:
                data = json.load(f)

            old_to_new_img = {}
            for img in data["images"]:
                old_id = img["id"]
                new_img = {
                    "id": img_id_counter,
                    "file_name": f"{prefix}_{img['file_name']}",
                    "width": img["width"],
                    "height": img["height"],
                }
                merged["images"].append(new_img)
                old_to_new_img[old_id] = img_id_counter
                img_id_counter += 1

                src = ds_path / split / "images" / img["file_name"]
                dst = out_split / "images" / f"{prefix}_{img['file_name']}"
                if src.exists():
                    shutil.copy2(src, dst)
                else:
                    print(f"  Warning: Missing {src}")

            for ann in data["annotations"]:
                merged["annotations"].append({
                    "id": ann_id_counter,
                    "image_id": old_to_new_img[ann["image_id"]],
                    "category_id": category_id,
                    "bbox": ann["bbox"],
                    "segmentation": ann.get("segmentation", []),
                    "area": ann["area"],
                    "iscrowd": ann.get("iscrowd", 0),
                })
                ann_id_counter += 1

            print(f"  {split} - {ds_config['category_name']}: {len(data['images'])} images")

        out_ann = out_split / "annotation.json"
        with open(out_ann, "w") as f:
            json.dump(merged, f, indent=2)

        print(
            f"  {split} TOTAL: {len(merged['images'])} images, "
            f"{len(merged['annotations'])} annotations"
        )
        print(f"  Categories: {[c['name'] for c in categories]}")

In [ ]:
#| export
CATEGORY_CONFIG = {
    "thin_wire": {"suffix": "_lblthin", "category_id": 1},
    "thick_wire": {"suffix": "_lblthick", "category_id": 1},
    "gate_pad": {"suffix": "_gatepads", "category_id": 1},
}

IMAGE_SUFFIX = "_original"

## Discover file groups

Group all files by their sample stem (everything before the category suffix).

In [ ]:
#| export
def discover_samples(src_dir: Path | str) -> dict[str, dict[str, Path]]:
    """
    Return {sample_stem: {'image': path, 'thin_wire': path, ...}}.
    """
    src_dir = Path(src_dir)
    groups: dict[str, dict[str, Path]] = defaultdict(dict)

    for p in sorted(src_dir.glob("*.png")):
        name = p.name
        if name.endswith(f"{IMAGE_SUFFIX}.png"):
            stem = name[: -len(f"{IMAGE_SUFFIX}.png")]
            groups[stem]["image"] = p
        else:
            for cat, cfg in CATEGORY_CONFIG.items():
                suffix = cfg["suffix"]
                if name.endswith(f"{suffix}.png"):
                    stem = name[: -len(f"{suffix}.png")]
                    groups[stem][cat] = p
                    break

    # Drop incomplete groups
    complete = {}
    required = {"image"} | set(CATEGORY_CONFIG)
    for stem, files in groups.items():
        if required.issubset(files):
            complete[stem] = dict(files)
        else:
            missing = required - set(files)
            print(f"Skipping incomplete sample {stem}, missing {missing}")

    return complete

In [ ]:
src_dir = Path(r'/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250512_parted_thin_flat')
src_dir.ls()

## Stage per-category splits

Copy images and masks into the layout expected by `coco_format.py`.
Train/test split is done at the sample level.

In [ ]:
#| export
def stage_clean_dataset(
    src_dir: Path | str,
    output_root: Path | str,
    test_size: float = 0.15,
    random_state: int = 42,
):
    """
    Stage the clean separated bondwire data into per-category train/test dirs.
    Returns the list of train stems and test stems.
    """
    src_dir = Path(src_dir)
    output_root = Path(output_root)
    samples = discover_samples(src_dir)
    stems = sorted(samples.keys())

    train_stems, test_stems = train_test_split(
        stems, test_size=test_size, random_state=random_state, shuffle=True
    )

    print(f"Total complete samples: {len(stems)}")
    print(f"Train: {len(train_stems)}, Test: {len(test_stems)}")

    for split, split_stems in [("train", train_stems), ("test", test_stems)]:
        for stem in tqdm(split_stems, desc=f"Staging {split}"):
            files = samples[stem]
            for cat in CATEGORY_CONFIG:
                img_dir = output_root / cat / split / "images"
                msk_dir = output_root / cat / split / "masks"
                img_dir.mkdir(parents=True, exist_ok=True)
                msk_dir.mkdir(parents=True, exist_ok=True)

                shutil.copy2(files["image"], img_dir / f"{stem}.png")
                shutil.copy2(files[cat], msk_dir / f"{stem}.png")

    return train_stems, test_stems

In [ ]:
train_stems, test_stems = stage_clean_dataset(
    src_dir=src_dir,
    output_root=Path(r'/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250512_cleaned'),
    test_size=0.15,
    random_state=42,
)
train_stems[:5], test_stems[:5]

In [ ]:
import matplotlib.pyplot as plt

## Convert each category/split to COCO

In [ ]:
#| export
def convert_to_coco(
    staging_root: Path | str,
    categories: list[str] | None = None,
    splits: list[str] = ["train", "test"],
) -> dict[str, dict[str, Path]]:
    """
    Run coco_format.py form_coco_dataset for every category + split.
    Returns {category: {split: annotation_path}}.
    """
    staging_root = Path(staging_root)
    categories = categories or list(CATEGORY_CONFIG.keys())
    paths: dict[str, dict[str, Path]] = {}

    for cat in categories:
        paths[cat] = {}
        for split in splits:
            img_dir = staging_root / cat / split / "images"
            msk_dir = staging_root / cat / split / "masks"
            out_dir = staging_root / cat / split

            coco = form_coco_dataset(
                mask_dir=msk_dir,
                img_dir=img_dir,
                categories=[{"id": 1, "name": cat}],
            )
            save_ann_file(out_dir, "annotation", coco)
            paths[cat][split] = out_dir / "annotation.json"
            print(f"{cat}/{split}: {len(coco['images'])} images, {len(coco['annotations'])} annotations")

    return paths

In [ ]:
converted_paths = convert_to_coco(
    staging_root=Path(r'/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250512_cleaned'),
    categories=list(CATEGORY_CONFIG.keys()),
    splits=["train", "test"],
)

In [ ]:
converted_paths

## Merge thick/thin/gate_pad into one dataset

Use the `merge_datasets_multi` helper from the repo's merge notebook.

In [ ]:
#| export
def merge_clean_bondwire(
    staging_root: Path | str,
    output_path: Path | str,
    prefixes: dict[str, str] | None = None,
):
    """
    Merge the three clean bondwire categories into one multi-class COCO dataset.
    """
    prefixes = prefixes or {"thick_wire": "g", "thin_wire": "g", "gate_pad": "g"}
    datasets = [
        {"path": str(Path(staging_root) / cat), "category_name": cat, "prefix": prefixes[cat]}
        for cat in ["thick_wire", "thin_wire", "gate_pad"]
    ]
    _merge_datasets_multi(datasets, str(output_path))

## Optional: patch into the existing 8-category dataset

If the existing `5_plus_bondwire_dataset` already contains the other categories and only the bondwire categories need replacing.

In [ ]:
#| export
def patch_into_existing(
    existing_dataset: Path | str,
    clean_staging: Path | str,
    output_dataset: Path | str,
):
    """
    Replace bondwire annotations in an existing merged dataset with clean ones.
    """
    import subprocess

    cmd = [
        "python",
        "-m",
        "sam3_fine_tune.preprocessing.patch_bondwire_annotations",
        "--src_dataset", str(existing_dataset),
        "--dst_dataset", str(output_dataset),
        "--bridged_staging", str(clean_staging),
        "--bondwire_categories", "thick_wire,thin_wire,gate_pad",
        "--bondwire_cat_ids", "6,7,8",
        "--src_prefix", "g_",
        "--bridged_prefixes", "g_,g_,g_",
    ]
    subprocess.run(cmd, check=True)

## Run the full pipeline

In [ ]:
# Adjust these paths for your HPC setup
SRC_DIR = Path("/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250512_parted_thin_flat")
STAGE_DIR = Path("/home/hpc_storage/data/projects/userB/sam3_data/bondwire_clean_staging")
MERGED_DIR = Path("/home/hpc_storage/data/projects/userB/sam3_data/bondwire_clean_merged")

# 1. Stage train/test per category
train_stems, test_stems = stage_clean_dataset(SRC_DIR, STAGE_DIR, test_size=0.15, random_state=42)

In [ ]:
# 2. Convert to COCO
coco_paths = convert_to_coco(STAGE_DIR)

In [ ]:
# 3. Merge into single 3-category dataset
merge_clean_bondwire(STAGE_DIR, MERGED_DIR)

## Why are there more annotations than images?

In COCO, each **connected component** in a mask becomes a separate annotation. A single image can contain many thick wires, thin wires, and gate pads, so the number of annotations is naturally larger than the number of images.

The next two cells inspect the merged dataset:

1. Print total image / annotation counts and a histogram of annotations-per-image.
2. Overlay annotations on a few sample images so you can visually confirm the conversion looks right.

If the overlays line up with the wires and pads you see in the images, the COCO conversion is working correctly.

### What to look for / common surprises

- **Multiple thick wires in one image**: Because each connected component becomes its own annotation, an image can have 8 (or more) `thick_wire` annotations if it contains 8 physically separate thick wires.
- **Bounding boxes that look empty**: COCO `bbox` is the axis-aligned bounding rectangle of every polygon pixel. A very thin diagonal wire can have a large empty area inside its bbox, so the box can look empty even though the wire pixels are inside it. The **contour overlay** (the colored mask) is the truth — the bbox is just a rough envelope.
- **Color washes out the image**: the default contour overlay uses a low alpha (transparency) so you can still see the underlying photo. Use `alpha=...` to make it more or less transparent.

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pycocotools.coco import COCO


def load_split_coco(merged_dir: Path, split: str = "train"):
    ann_path = Path(merged_dir) / split / "annotation.json"
    if not ann_path.exists():
        raise FileNotFoundError(
            f"{ann_path} not found. Run the merge step above first."
        )
    return COCO(str(ann_path))


coco_train = load_split_coco(MERGED_DIR, "train")
coco_test = load_split_coco(MERGED_DIR, "test")

cat_id_to_name = {c["id"]: c["name"] for c in coco_train.dataset["categories"]}


def count_summary(coco, split: str):
    img_ids = coco.getImgIds()
    counts = []
    per_cat = defaultdict(int)
    for img_id in img_ids:
        ann_ids = coco.getAnnIds(imgIds=[img_id])
        counts.append(len(ann_ids))
        for ann in coco.loadAnns(ann_ids):
            per_cat[cat_id_to_name[ann["category_id"]]] += 1
    print(f"\n=== {split} ===")
    print(f"Images: {len(img_ids)}")
    print(f"Total annotations: {sum(counts)}")
    print(
        f"Annotations per image: mean={np.mean(counts):.2f}, "
        f"max={max(counts) if counts else 0}, min={min(counts) if counts else 0}"
    )
    print("Per-category annotations:")
    for name in sorted(per_cat):
        print(f"  {name}: {per_cat[name]}")
    return counts


train_counts = count_summary(coco_train, "train")
test_counts = count_summary(coco_test, "test")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(
    train_counts, bins=range(0, max(train_counts or [0]) + 2), align="left", rwidth=0.8
)
axes[0].set_title("Train: annotations per image")
axes[0].set_xlabel("# annotations")
axes[0].set_ylabel("# images")

axes[1].hist(
    test_counts, bins=range(0, max(test_counts or [0]) + 2), align="left", rwidth=0.8
)
axes[1].set_title("Test: annotations per image")
axes[1].set_xlabel("# annotations")
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import matplotlib.patches as patches

COLORS = {
    "thick_wire": (0.65, 0.12, 0.51),  # berry
    "thin_wire": (0.04, 0.51, 0.47),   # ocean
    "gate_pad": (0.95, 0.46, 0.08),    # sun
}


def overlay_annotations(
    coco,
    split: str,
    n_samples: int = 3,
    seed: int = 42,
    show_contours: bool = True,
    show_boxes: bool = False,
    show_labels: bool = False,
    alpha: float = 0.35,
):
    rng = random.Random(seed)
    img_ids = coco.getImgIds()
    chosen = rng.sample(img_ids, min(n_samples, len(img_ids)))
    fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 5))
    if n_samples == 1:
        axes = [axes]

    for ax, img_id in zip(axes, chosen):
        img_info = coco.loadImgs(img_id)[0]
        img_path = MERGED_DIR / split / "images" / img_info["file_name"]
        img = np.array(Image.open(img_path).convert("RGB"))
        ax.imshow(img)

        ann_ids = coco.getAnnIds(imgIds=[img_id])
        anns = coco.loadAnns(ann_ids)
        title_parts = defaultdict(int)

        for ann_idx, ann in enumerate(anns):
            name = cat_id_to_name[ann["category_id"]]
            title_parts[name] += 1
            color = np.array(COLORS.get(name, (0.0, 0.0, 0.0)))
            mask = coco.annToMask(ann)

            if show_contours:
                overlay = (mask[..., None] * color).astype(float)
                ax.imshow(overlay, alpha=alpha)

            if show_boxes:
                x, y, w, h = ann["bbox"]
                ax.add_patch(
                    patches.Rectangle(
                        (x, y), w, h, linewidth=1.5, edgecolor=color, facecolor="none"
                    )
                )

            if show_labels:
                ys, xs = np.where(mask)
                if len(xs) > 0:
                    ax.text(
                        xs[0],
                        ys[0],
                        f"{ann_idx + 1}",
                        color=color,
                        fontsize=8,
                        bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1),
                    )

        title = ", ".join(f"{k}={v}" for k, v in sorted(title_parts.items()))
        ax.set_title(f"{img_info['file_name']}\n{title}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# Default overlay: contours only, low alpha so the image stays visible
overlay_annotations(coco_train, "train", n_samples=3, alpha=0.35)
overlay_annotations(coco_test, "test", n_samples=3, alpha=0.35)

# Diagnostic view: also show bounding boxes + annotation indices to investigate empty boxes
overlay_annotations(coco_train, "train", n_samples=3, seed=123, show_boxes=True, show_labels=True, alpha=0.25)
overlay_annotations(coco_test, "test", n_samples=3, seed=123, show_boxes=True, show_labels=True, alpha=0.25)

## Verify round-trip (per-category)

For each category, reconstruct masks from COCO and compare against the source binary masks. Expect IoU ≈ 1.0 because the source labels are already clean.

In [ ]:
from sam3_fine_tune.preprocessing.verify_coco_roundtrip import verify_roundtrip

for cat in ["thick_wire", "thin_wire", "gate_pad"]:
    for split in ["train", "test"]:
        print(f"\n=== {cat} {split} ===")
        src_mask_dir = STAGE_DIR / cat / split / "masks"
        coco_json = STAGE_DIR / cat / split / "annotation.json"
        verify_roundtrip(
            coco_json=coco_json,
            src_mask_dir=src_mask_dir,
            label_map={1: cat},
            priority_order=[1],
            mask_suffix="",
            per_category=True,
        )
        # Remove this break to verify all categories and splits

## Optional: replace bondwire annotations in the existing 8-category dataset

Use this only if you want to keep the existing et_pin / eberhard_pin / wuxi_pin / war_crack / matrix_chip images and replace only the bondwire annotations.

In [ ]:
# EXISTING_DATASET = Path("/home/hpc_storage/data/projects/userB/sam3_data/5_plus_bondwire_dataset")
# OUTPUT_DATASET = Path("/home/hpc_storage/data/projects/userB/sam3_data/5_plus_bondwire_dataset_clean")
# patch_into_existing(EXISTING_DATASET, STAGE_DIR, OUTPUT_DATASET)

In [ ]:
#| export
def save_overlay_poster(
    coco_json: Path,
    image_dir: Path | None = None,
    output_path: Path | None = None,
    n_samples: int = 4,
    seed: int = 42,
    show_boxes: bool = False,
    show_labels: bool = False,
    alpha: float = 0.45,
    dpi: int = 200,
):
    """Save a JPEG poster with 3 panels per image: original, overlay, mask."""
    coco_json = Path(coco_json)
    if image_dir is None:
        image_dir = coco_json.parent / "images"
    image_dir = Path(image_dir)

    if output_path is None:
        output_path = coco_json.parent / "overlay_poster.jpg"
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    coco = COCO(str(coco_json))
    cat_id_to_name = {c["id"]: c["name"] for c in coco.dataset["categories"]}

    rng = random.Random(seed)
    img_ids = coco.getImgIds()
    chosen = rng.sample(img_ids, min(n_samples, len(img_ids)))

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    axes = np.atleast_2d(axes)

    for row, img_id in enumerate(chosen):
        img_info = coco.loadImgs(img_id)[0]
        img_path = image_dir / img_info["file_name"]
        img = np.array(Image.open(img_path).convert("RGB"))

        ax_orig, ax_overlay, ax_mask = axes[row]

        # 1) Original image
        ax_orig.imshow(img)
        ax_orig.set_title("original")
        ax_orig.axis("off")

        # 2) Overlay image
        ax_overlay.imshow(img)

        ann_ids = coco.getAnnIds(imgIds=[img_id])
        anns = coco.loadAnns(ann_ids)
        title_parts = defaultdict(int)

        mask_canvas = np.zeros((*img.shape[:2], 3), dtype=np.uint8)
        for ann_idx, ann in enumerate(anns):
            name = cat_id_to_name[ann["category_id"]]
            title_parts[name] += 1
            color = (np.array(COLORS.get(name, (0.0, 0.0, 0.0))) * 255).astype(np.uint8)
            mask = coco.annToMask(ann).astype(bool)
            mask_canvas[mask] = color

            if show_boxes:
                x, y, w, h = ann["bbox"]
                ax_overlay.add_patch(
                    patches.Rectangle(
                        (x, y), w, h, linewidth=1.5, edgecolor=color / 255.0, facecolor="none"
                    )
                )

            if show_labels:
                ys, xs = np.where(mask)
                if len(xs) > 0:
                    ax_overlay.text(
                        xs[0],
                        ys[0],
                        f"{ann_idx + 1}",
                        color=color / 255.0,
                        fontsize=8,
                        bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1),
                    )

        ax_overlay.imshow(mask_canvas, alpha=alpha)
        title = ", ".join(f"{k}={v}" for k, v in sorted(title_parts.items()))
        ax_overlay.set_title(f"overlay\n{title}")
        ax_overlay.axis("off")

        # 3) Mask only
        ax_mask.imshow(mask_canvas)
        ax_mask.set_title("mask only")
        ax_mask.axis("off")

    plt.tight_layout()
    fig.savefig(output_path, format="jpeg", dpi=dpi, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved poster to: {output_path}")
    return output_path

In [ ]:
# Save a JPEG poster to verify the merged COCO conversion
save_overlay_poster(
    coco_json=MERGED_DIR / "train" / "annotation.json",
    image_dir=MERGED_DIR / "train" / "images",
    output_path=MERGED_DIR / "overlay_poster_train.jpg",
    n_samples=4,
    seed=42,
    alpha=0.35,
    dpi=150,
    ncols=2,
)

save_overlay_poster(
    coco_json=MERGED_DIR / "test" / "annotation.json",
    image_dir=MERGED_DIR / "test" / "images",
    output_path=MERGED_DIR / "overlay_poster_test.jpg",
    n_samples=4,
    seed=42,
    alpha=0.35,
    dpi=150,
    ncols=2,
)


In [ ]:
# Panda Bent wire data

In [ ]:
data_path = Path(r'/home/hpc_storage/data/projects/userB/data/universal_model/ThinWire_Segmentation')
Categories = ['thin_wire', 'thick_wire', 'gate_pad']
data_path.ls()